# 第 29 章 · 为什么融合能变快：算力受限与访存受限

本章回答：同样是一个 kernel，为什么矩阵乘法"算得满"、加法和 GELU 却"算不满"？roofline（屋顶线）图怎么读？融合到底省了什么，能快多少？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/29-why-fusion.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch13「GEMM 的内核 Roofline 分析」（`ch13.md` 第 381–444 行）；Ultra-Scale Playbook §How to improve performance with Kernels / Fused Kernels（`ultrascale-playbook.md` 第 2405 行、第 2711 行）；CUDA 讲义 [第 9 章 roofline](../../../cuda-programming/notes/lessons/09-roofline-profiling.md)；Williams et al. 2009「Roofline」（〔在线〕）；本章计时是本机 Apple M 系列 GPU（MPS）上 torch 2.15.0 实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import time

import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 拐点 | 峰值 ÷ 带宽 | `ridge(989e12, 3.35e12)` | 295 |
| 第二步 算术强度 | 1/12；n/3 | `intensity_matmul(4096)` | 0.083；1365 |
| 第三步 融合实测 | 8 个张量 vs 4 个 | `fusion_benchmark()` | 约 1.1 ms vs 0.56 ms，带宽都约 480 GB/s |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `f`**（来自 `ch29_why_fusion.py`）：下面要用到的小工具。

In [ ]:
def f(x, b, s):
    return F.gelu(x + b) * s

**函数 `fusion_benchmark`**（来自 `ch29_why_fusion.py`）：返回 [(名字, 毫秒, 读写字节, 有效 GB/s)]：eager 读写 8 个张量，compile 4 个。

In [ ]:
def fusion_benchmark(n: int = 1 << 24, reps: int = 20):
    """返回 [(名字, 毫秒, 读写字节, 有效 GB/s)]：eager 读写 8 个张量，compile 4 个。"""
    dev = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    sync = {"mps": torch.mps.synchronize, "cuda": torch.cuda.synchronize}.get(dev, lambda: None)
    x, b, s = (torch.randn(n, device=dev) for _ in range(3))
    out = []
    for name, fn, tensors in (("eager", f, 8), ("compile", torch.compile(f), 4)):
        for _ in range(5):
            fn(x, b, s)
        sync()
        t = time.perf_counter()
        for _ in range(reps):
            fn(x, b, s)
        sync()
        ms = (time.perf_counter() - t) / reps * 1e3
        nbytes = tensors * n * 4
        out.append((name, round(ms, 2), nbytes, round(nbytes / ms / 1e6)))
    return dev, out

**函数 `intensity_matmul`**（来自 `ch29_why_fusion.py`）：下面要用到的小工具。

In [ ]:
def intensity_matmul(n: int, bytes_per: int = 2) -> float:
    return 2 * n**3 / (3 * n * n * bytes_per)

**函数 `ridge`**（来自 `ch29_why_fusion.py`）：下面要用到的小工具。

In [ ]:
def ridge(peak_flops: float, bandwidth: float) -> float:
    return peak_flops / bandwidth

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 H100 拐点 ≈", round(ridge(989e12, 3.35e12)), "FLOP/字节")

In [ ]:
print("\n第二步 fp32 add 算术强度 1/12 ≈", round(1 / 12, 3), "；矩阵乘 n=4096（bf16）≈", round(intensity_matmul(4096)))

In [ ]:
print("\n第三步 gelu(x+b)*s 融合前后：", fusion_benchmark())

## 3. 代码里才看得到的

1. **MPS 上 `torch.compile` 生成的是 Metal 代码**：第 31 章能看到。
2. **没有 GPU 时退回 CPU**：比例还在，但带宽数字小得多。

## 4. 自己改一改

- 把 f 改成 `F.gelu(x + b)`（2 个算子），预测 eager 读写几个张量再跑。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 时间看两笔：算的和搬的，谁大谁是瓶颈；算术强度低于拐点就是访存受限；
> 矩阵乘吃算力，其余算子吃带宽，时间只看读写字节；
> 融合不让中间结果落显存，带宽不变字节减半时间减半；手写之前先算离理论最快还差多少。